# GCS Sharding Dataset con PyTorch

Questo notebook carica il dataset Iris da Google Cloud Storage (GCS), lo divide in shard (frammenti), e crea un DataLoader lazy che scarica i dati on-demand dai shard.

## Workflow
1. **Autenticazione** → Login con credenziali Google
2. **Accesso al bucket** → Leggi file da GCS
3. **Preprocessing** → Carica CSV e converti in tensori PyTorch
4. **Sharding** → Dividi il dataset e carica su GCS
5. **Dataset Lazy** → Leggi shard su richiesta durante l'addestramento

## Step 1: Autenticazione con Google Cloud

Autentica il tuo account Google per accedere ai bucket GCS con le credenziali locali salvate da `gcloud`.

In [ ]:
!gcloud auth application-default login

## Step 2: Configurazione del Bucket GCS

Specifica il nome del bucket GCS dove sono salvati i dati.

In [ ]:
NOME_DEL_BUCKET="ias-luigi-asprino-bucket"

## Step 3: Accesso al Bucket con gcsfs

`gcsfs` usa le credenziali autenticate (`google_default`) per elencare i file nel bucket.

In [ ]:
# Cella 2: accesso al bucket con gcsfs
import gcsfs

fs = gcsfs.GCSFileSystem(token="google_default")
files = fs.ls(f'gs://{NOME_DEL_BUCKET}/')   # lista gli oggetti
print(files)

## Step 4: Caricamento del Dataset

Leggi il file CSV `iris.csv` dal bucket usando `gcsfs.open()` e carica in un pandas DataFrame.

In [ ]:
import pandas, io
# Leggere un CSV
with fs.open(f'gs://{NOME_DEL_BUCKET}/iris.csv', 'rb') as f:
    df = pandas.read_csv(io.BytesIO(f.read()))

print(f"Dataset caricato: {len(df)} righe, colonne: {list(df.columns)}")


## Step 5: Analisi Esplorativa

Analizza i tipi di dato, colonne categoriche, valori mancanti e visualizza le prime righe.

In [ ]:
# Controlla i tipi di ogni colonna
print(df.dtypes)

# Identifica le colonne object
print(df.select_dtypes(include='object').columns.tolist())

# Controlla i valori mancanti
print(df.isnull().sum())

# Guarda le prime righe delle colonne problematiche
print(df.head())

## Step 6: Preprocessing - Encoding delle Variabili Categoriche

La colonna `variety` è categorica. La convertiamo in numeri interi (0, 1, 2).

In [ ]:
df['variety'] = df['variety'].astype('category').cat.codes

## Step 7: Conversione in Tensori PyTorch

Converti il DataFrame in tensori PyTorch:
- **X**: feature (dimensioni 4) come float32
- **y**: etichette come valori interi (long)

In [ ]:
import torch
# ── 3. CONVERTI IN TENSORI ───────────────────────────────────────────────────
X = torch.tensor(df.iloc[:, :-1].values, dtype=torch.float32)  # feature → float
y = torch.tensor(df.iloc[:,  -1].values, dtype=torch.long)     # label   → int

## Step 8: Sharding e Upload su GCS

Dividi il dataset in **4 shard** (frammenti) e carica ogni shard come file `.pt` su GCS.

Ogni shard contiene:
- `chunk['X']`: tensore con le feature dello shard
- `chunk['y']`: tensore con le etichette dello shard

In [ ]:
import math

# ── 4. CREA E CARICA GLI SHARD SU GCS ────────────────────────────────────────
NUM_SHARDS = 4   # con dataset piccoli bastano pochi shard; adatta a num_workers
shard_size = math.ceil(len(X) / NUM_SHARDS)

for i in range(NUM_SHARDS):
    start = i * shard_size
    end   = min(start + shard_size, len(X))      # l'ultimo shard può essere più corto

    chunk = {
        'X': X[start:end],   # tensore [shard_size, 4]
        'y': y[start:end],   # tensore [shard_size]
    }

    uri = f'gs://{NOME_DEL_BUCKET}/shards/shard-{i:03d}.pt'
    with fs.open(uri, 'wb') as f:
        torch.save(chunk, f)

    print(f"Shard {i:03d}: righe {start}–{end-1}  ({len(chunk['X'])} campioni) → {uri}")

## Step 9: Verifica dello Shard

Leggi il primo shard da GCS e verifica le dimensioni del tensore.

In [ ]:
with gcsfs.GCSFileSystem(token="google_default").open("gs://ias-luigi-asprino-bucket/shards/shard-000.pt", 'rb') as f:
    chunk = torch.load(io.BytesIO(f.read()))

print(chunk['X'].shape)
print(chunk['y'].shape)

## Step 10: Dataset Lazy con Streaming da GCS

La classe `GCSShardDataset` implementa `torch.utils.data.Dataset`:
- **Lazy loading**: carica i dati solo quando richiesti (non all'inizializzazione)
- **Caching per worker**: ogni processo worker mantiene in memoria gli ultimi 2 shard scaricati
- **Streaming**: scarica i shard da GCS on-demand durante l'addestramento

La classe è salvata nel file `gcpbucketdataset.py` per evitare errori di pickling con `num_workers > 0`.

## Step 11: Creazione del DataLoader

Crea un `DataLoader` con:
- **batch_size**: metà della dimensione dello shard
- **num_workers**: 2 processi worker per parallelizzare il caricamento dei dati
- Ogni worker gestisce il proprio cache locale degli shard

Il loop di addestramento itera su epoch e batch, caricando i dati on-demand da GCS.

In [ ]:
# Parametri — devono corrispondere a quelli usati durante lo sharding
from gcpbucketdataset import GCSShardDataset
from torch.utils.data import DataLoader

dataset = GCSShardDataset(NOME_DEL_BUCKET, NUM_SHARDS, shard_size, total_samples=len(X))

dl = DataLoader(
    dataset,
    batch_size  = shard_size // 2,
    num_workers = 2
)

# Training loop
for epoch in range(2):
    print(f"Epoch {epoch}")
    for X_batch, y_batch in dl:
        print(f"epoch: {epoch} X: {len(X_batch)} Y: {len(y_batch)}")
        # forward / backward ...
